In [1]:
import emap
import json
import time

TEST_NAME = "adder"
TOP_MODULE = "eval/epfl/adder"
MAX_ITER = 4

start_time = time.time()

netlist = emap.NetlistDB(schema_file="emap/schema.sql", cnt=10000)
netlist.VERBOSE = True
with open(f"eval/epfl/{TEST_NAME}.json") as f:
    netlist.build_from_json(json.load(f)["modules"][TOP_MODULE])

wdsu = emap.DisjointSetUnion()
netlist.rebuild(wdsu)

for i in range(MAX_ITER):
    matches0 = emap.rewrites.ematch_not_idemp(netlist)
    matches1 = emap.rewrites.ematch_and_idemp(netlist)
    matches2 = emap.rewrites.ematch_and_assoc_left(netlist)
    matches3 = emap.rewrites.ematch_and_comm(netlist)
    matches4 = emap.rewrites.ematch_and_comp(netlist)

    cnt = 0
    cnt += emap.rewrites.apply_not_idemp(matches0, wdsu)
    cnt += emap.rewrites.apply_and_idemp(matches1, wdsu)
    cnt += emap.rewrites.apply_and_assoc_left(netlist, matches2)
    cnt += emap.rewrites.apply_and_comm(netlist, matches3)
    cnt += emap.rewrites.apply_and_comp(matches4, wdsu)

    if cnt > 0:
        print(f"Applied {cnt} rewrites")
        netlist.rebuild(wdsu)
    else:
        print("No more rewrites can be applied. Stopping.")
        break

# lut map
emap.rewrites.techmap_luts(netlist, k=6, cnt=100, rseed=42)

with open("debug.json", "w") as f:
    json.dump(netlist.dump_tables(), f, indent=2)

with open(f"eval/out/saturated_{TEST_NAME}.json", "w") as f:
    json.dump({"creator": "nextmap", "modules": {"top": netlist.write_json()}}, f, indent=2)

Found 2411 cells
Processing cell 0/2411: $and$eval/epfl/adder_aig.v:5121$257
Processing cell 1000/2411: $and$eval/epfl/adder_aig.v:6822$1958
Processing cell 2000/2411: $not$eval/epfl/adder_aig.v:6283$1419
Applied 1435 rewrites
Applied 1312 rewrites
Applied 2569 rewrites
Updating cells for wire 0/340
Rebuild iteration 0 done.
Applied 1940 rewrites
Updating cells for wire 0/500
Rebuild iteration 0 done.
Techmapping to 6-LUTs with random seed 42
Chosen wire 2354 with shared times 27
  Cone: {240, 112}
Chosen wire 465 with shared times 27
  Cone: {7, 135}
Chosen wire 1237 with shared times 30
  Cone: {178, 50}
Chosen wire 1075 with shared times 30
  Cone: {169, 41}
Chosen wire 10006 with shared times 2
  Cone: {10530, 10543, 470, 10533, 10540, 543}
Chosen wire 2468 with shared times 13
  Cone: {11072, 2376, 11052, 11059, 11062, 11069}
Chosen wire 10981 with shared times 2
  Cone: {10977, 10964, 2054, 10967, 10974, 11951}
Chosen wire 656 with shared times 2
  Cone: {645, 10566, 10570, 144, 